In [1]:
# 1. Giải nén file rawdata.zip
!unzip -q -o rawdata.zip

# 2. Tách data y khoa (Tự động gom dữ liệu từ các file txt)
import pandas as pd
import os
import glob

OUTPUT_DIR = "processed_data/extracted_features"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_PARAMS = ["Cholesterol", "RespRate", "Creatinine", "NIMAP"]
data_dict = {param: [] for param in TARGET_PARAMS}

# Tìm tất cả các file txt trong thư mục giải nén
files = glob.glob("**/*.txt", recursive=True)
files = [f for f in files if "Outcomes" not in f] # Loại trừ file nhãn chung

print(f"[*] Đang lùng sục trong {len(files)} file bệnh nhân...")

for fname in files:
    record_id = os.path.basename(fname).replace(".txt", "")
    try:
        df = pd.read_csv(fname)
        for param in TARGET_PARAMS:
            temp_df = df[df["Parameter"] == param].copy()
            if not temp_df.empty:
                temp_df.insert(0, "RecordID", record_id)
                data_dict[param].append(temp_df)
    except:
        pass

print("\n[*] KẾT QUẢ TÁCH DATA:")
for param in TARGET_PARAMS:
    if len(data_dict[param]) > 0:
        final_df = pd.concat(data_dict[param], ignore_index=True)
        output_path = os.path.join(OUTPUT_DIR, f"{param}.csv")
        final_df.to_csv(output_path, index=False)
        print(f"✅ Đã lưu {param} -> {output_path} (Tổng: {len(final_df)} dòng)")

print("\n🎉 Hoàn tất! Cấu trúc folder đã chuẩn bị sẵn sàng cho SLM")

[*] Đang lùng sục trong 12000 file bệnh nhân...

[*] KẾT QUẢ TÁCH DATA:
✅ Đã lưu Cholesterol -> processed_data/extracted_features/Cholesterol.csv (Tổng: 995 dòng)
✅ Đã lưu RespRate -> processed_data/extracted_features/RespRate.csv (Tổng: 165435 dòng)
✅ Đã lưu Creatinine -> processed_data/extracted_features/Creatinine.csv (Tổng: 41928 dòng)
✅ Đã lưu NIMAP -> processed_data/extracted_features/NIMAP.csv (Tổng: 290496 dòng)

🎉 Hoàn tất! Cấu trúc folder đã chuẩn bị sẵn sàng cho SLM


In [ ]:
# 1. Gỡ bỏ TensorFlow tránh xung đột (Sửa lỗi Protobuf)
!pip uninstall -y tensorflow tensorboard opentelemetry-api opentelemetry-sdk

# 2. Cài đặt "Combo Vàng" cho vLLM
!pip install "vllm==0.8.4" "transformers==4.57.0" pandas numpy scikit-learn scipy huggingface_hub nest_asyncio -q
from huggingface_hub import login
login() # Nó sẽ tự động hiện một ô input an toàn để bạn dán token vào lúc chạy

# 3. Bẻ khóa Float16
import os, re, importlib.util
try:
    vllm_dir = importlib.util.find_spec("vllm").submodule_search_locations[0]
    for root, dirs, files in os.walk(vllm_dir):
        for file in files:
            if file.endswith(".py"):
                path = os.path.join(root, file)
                with open(path, "r", encoding="utf-8", errors="ignore") as f: content = f.read()
                if "_FLOAT16_NOT_SUPPORTED_MODELS" in content:
                    with open(path, "w", encoding="utf-8") as f:
                        f.write(re.sub(r'_FLOAT16_NOT_SUPPORTED_MODELS\s*=\s*\{[^}]*\}', '_FLOAT16_NOT_SUPPORTED_MODELS = set()', content))
    print("✅ Đã bẻ khóa Float16 thành công!")
except Exception as e: print(e)

# 4. Môi trường an toàn
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"
os.environ["NCCL_P2P_DISABLE"] = "1"
print("✅ Đã thiết lập môi trường an toàn cho Colab!")

Found existing installation: opentelemetry-api 1.26.0
Uninstalling opentelemetry-api-1.26.0:
  Successfully uninstalled opentelemetry-api-1.26.0
Found existing installation: opentelemetry-sdk 1.26.0
Uninstalling opentelemetry-sdk-1.26.0:
  Successfully uninstalled opentelemetry-sdk-1.26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.26.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.26.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.26.0 which is incompatible.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.2.1 which is incompatible.
opentelemetry-exporter-gcp-

In [3]:
!pip install "bitsandbytes>=0.45.3"

In [ ]:
# -*- coding: utf-8 -*-
"""
vllm_nimap_forecast.py - Batch-Retry vLLM cho NIMAP Forecasting.
Chay 6 temperature mot luot, model load 1 lan.
"""

import pandas as pd
import numpy as np
import re
import warnings
from sklearn.metrics import mean_absolute_error, mean_squared_error
from vllm import LLM, SamplingParams

warnings.filterwarnings("ignore")

# ==========================================
# 1. CẤU HÌNH
# ==========================================
# HF_MODEL_ID = "unsloth/gemma-2-2b-it"
# HF_MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"
HF_MODEL_ID = "microsoft/Phi-3.5-mini-instruct"
# HF_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
# HF_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

FILE_PATH = "processed_data/extracted_features/NIMAP.csv"
OUTPUT_CSV = f"Temp_Results_{HF_MODEL_ID.replace('/', '_')}.csv"

LOOKBACK = 10
N_STEPS = 1
MAX_PATIENTS = 10
SEED = 42

ATTEMPTS_PER_STYLE = 2
MAX_BATCH_ITER = 15

TEMP_LIST = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7,0.8, 0.9]

# ==========================================
# 2. KHỞI TẠO ENGINE vLLM (1 LAN DUY NHAT)
# ==========================================
print(f"⏳ Đang tải mô hình {HF_MODEL_ID} vào vLLM Engine...")
import transformers
if not hasattr(transformers.PretrainedConfig, "sliding_window_pattern"):
    transformers.PretrainedConfig.sliding_window_pattern = 4096
llm_kwargs = {
    "model": HF_MODEL_ID,
    "trust_remote_code": True,
    "max_model_len": 4096,
    "gpu_memory_utilization": 0.85,
    "dtype": "half",
    "enforce_eager": True,
    "tensor_parallel_size": 1
}

try:
    llm = LLM(**llm_kwargs)
    print("✅ vLLM Engine đã sẵn sàng!")
except Exception as e:
    raise RuntimeError(f"❌ Lỗi khởi tạo vLLM: {e}")

# ==========================================
# 3. PROMPT + PARSER
# ==========================================
def calc_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    if not mask.any(): return 0.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def parse_irregular_output(text: str, horizon: int):
    if not text:
        return None
    numbers = re.findall(r'\b(\d+\.?\d*)\b', text)
    result = []
    for n in numbers:
        try:
            val = float(n)
            if 30 < val < 200:
                result.append(val)
                if len(result) >= horizon:
                    return result
        except:
            continue
    return result if len(result) >= horizon else None

def build_prompt(history_data: pd.DataFrame, style: int) -> str:
    vals = [round(float(v), 1) for v in history_data['Value'].tolist()]
    if style == 0:
        return f"""Given these NIMAP blood pressure values: {vals}
What is the next value? Reply with ONLY a number, nothing else."""
    else:
        return f"""NIMAP values: {vals}
Predict next value. Single number only."""

# ==========================================
# 4. QUẢN LÝ TÁC VỤ
# ==========================================
class WindowTask:
    def __init__(self, record_id, window_idx, history_df, actual_df):
        self.record_id = record_id
        self.window_idx = window_idx
        self.history_df = history_df
        self.actual_df = actual_df
        self.actual_values = actual_df['Value'].tolist()
        self.task_id = f"{record_id}_W{window_idx}"

        self.total_attempts = 0
        self.style = 0
        self.attempt_in_style = 0
        self.success = False
        self.predicted_values = []

    def get_sampling_params(self, temp):
        return SamplingParams(
            temperature=temp,
            top_p=1.0 if temp == 0 else 0.9,
            max_tokens=50,
            repetition_penalty=1.1
        )

    def advance_state(self):
        self.total_attempts += 1
        self.attempt_in_style += 1
        if self.attempt_in_style >= ATTEMPTS_PER_STYLE:
            self.attempt_in_style = 0
            self.style = (self.style + 1) % 2

# ==========================================
# 5. MAIN - CHAY 6 TEMPERATURE MOT LUOT
# ==========================================
def main():
    print(f"\n{'═'*65}")
    print(f"  DATA   : {FILE_PATH}")
    print(f"  MODEL  : {HF_MODEL_ID}")
    print(f"  LB={LOOKBACK}, Forecast={N_STEPS}")
    print(f"  Temperatures: {TEMP_LIST}")
    print(f"{'═'*65}")

    df = pd.read_csv(FILE_PATH)
    df = df.sort_values(by=['RecordID', 'Time']).reset_index(drop=True)

    # Patient-level split (70/15/15)
    rng = np.random.RandomState(SEED)
    all_patients = df['RecordID'].unique()
    rng.shuffle(all_patients)
    n = len(all_patients)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    test_patients = all_patients[n_train + n_val:]
    patient_ids = test_patients[:MAX_PATIENTS] if MAX_PATIENTS else test_patients
    print(f"    Tong benh nhan: {n}, Test: {len(test_patients)}, Using: {len(patient_ids)}")

    # Tao windows 1 lan duy nhat
    all_windows = []
    for pid in patient_ids:
        pdata = df[df['RecordID'] == pid].reset_index(drop=True)
        if len(pdata) < (LOOKBACK + N_STEPS):
            continue
        for i in range(0, len(pdata) - LOOKBACK - N_STEPS + 1):
            history_df = pdata.iloc[i : i + LOOKBACK]
            actual_df = pdata.iloc[i + LOOKBACK : i + LOOKBACK + N_STEPS]
            all_windows.append((pid, i, history_df, actual_df))
    print(f"    Tong windows: {len(all_windows)}")

    all_temp_results = []

    # LOOP QUA TUNG TEMPERATURE
    for temp in TEMP_LIST:
        print(f"\n{'─'*65}")
        print(f"  🌡️ Temperature = {temp}")
        print(f"{'─'*65}")

        # Tao tasks moi cho moi temperature
        tasks = [WindowTask(pid, idx, h_df, a_df) for pid, idx, h_df, a_df in all_windows]

        batch_iteration = 1
        while any(not t.success for t in tasks):
            if batch_iteration > MAX_BATCH_ITER:
                for t in tasks:
                    if not t.success:
                        t.predicted_values = [float(t.history_df['Value'].iloc[-1])]
                        t.success = True
                break

            active_tasks = [t for t in tasks if not t.success]
            print(f"  [Batch {batch_iteration}] {len(active_tasks)} windows...", end=" ")

            batch_prompts = [build_prompt(t.history_df, t.style) for t in active_tasks]
            batch_params = [t.get_sampling_params(temp) for t in active_tasks]
            outputs = llm.generate(prompts=batch_prompts, sampling_params=batch_params, use_tqdm=False)

            success_in_batch = 0
            for t, output in zip(active_tasks, outputs):
                raw_text = output.outputs[0].text.strip()
                preds = parse_irregular_output(raw_text, N_STEPS)
                if preds is not None:
                    t.success = True
                    t.predicted_values = preds
                    success_in_batch += 1
                else:
                    t.advance_state()

            print(f"-> {success_in_batch} OK, {len(active_tasks)-success_in_batch} retry")
            batch_iteration += 1

        # Tinh metric
        success_tasks = [t for t in tasks if t.success]
        all_maes, all_mapes, all_rmses, all_das = [], [], [], []

        for t in success_tasks:
            mae = mean_absolute_error(t.actual_values, t.predicted_values)
            rmse = np.sqrt(mean_squared_error(t.actual_values, t.predicted_values))
            mape = calc_mape(t.actual_values, t.predicted_values)
            last_obs = t.history_df['Value'].iloc[-1]
            actual_dir = np.sign(np.array(t.actual_values) - last_obs)
            pred_dir = np.sign(np.array(t.predicted_values) - last_obs)
            da = float(np.mean(actual_dir == pred_dir) * 100)

            all_maes.append(mae)
            all_mapes.append(mape)
            all_rmses.append(rmse)
            all_das.append(da)

        maxae_all = max(abs(t.actual_values[0] - t.predicted_values[0]) for t in success_tasks)
        fallback_count = sum(1 for t in tasks if t.total_attempts >= MAX_BATCH_ITER)

        row = {
            'Model': HF_MODEL_ID,
            'Lookback': LOOKBACK,
            'Temperature': temp,
            'MAE': round(np.median(all_maes), 4),
            'RMSE': round(np.median(all_rmses), 4),
            'MAPE (%)': f"{round(np.mean(all_mapes), 2)}%",
            'MaxAE': round(maxae_all, 2),
            'DA (%)': f"{round(np.mean(all_das), 2)}%",
            'Fallback': f"{fallback_count}/{len(tasks)}",
        }
        all_temp_results.append(row)

        print(f"    MAE={row['MAE']} | MAPE={row['MAPE (%)']} | DA={row['DA (%)']} | Fallback={row['Fallback']}")

    # Xuat bang tong hop
    results_df = pd.DataFrame(all_temp_results)
    results_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n{'═'*65}")
    print(f"  BANG TONG HOP TEMPERATURE - {HF_MODEL_ID}")
    print(f"{'═'*65}")
    print(results_df.to_string(index=False))
    print(f"\n[+] Da luu: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

⏳ Đang tải mô hình microsoft/Phi-3.5-mini-instruct vào vLLM Engine...


config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 06-05 04:58:34 [config.py:2836] Casting torch.bfloat16 to torch.float16.
WARNING 06-05 04:58:54 [arg_utils.py:1731] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
WARNING 06-05 04:58:54 [cuda.py:96] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used


...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


✅ vLLM Engine đã sẵn sàng!

═════════════════════════════════════════════════════════════════
  DATA   : processed_data/extracted_features/NIMAP.csv
  MODEL  : microsoft/Phi-3.5-mini-instruct
  LB=10, Forecast=1
  Temperatures: [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
═════════════════════════════════════════════════════════════════
    Tong benh nhan: 10464, Test: 1571, Using: 10
    Tong windows: 210

─────────────────────────────────────────────────────────────────
  🌡️ Temperature = 0
─────────────────────────────────────────────────────────────────
  [Batch 1] 210 windows... WARNING 06-05 05:03:48 [scheduler.py:1769] Sequence group 136 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1
-> 0 OK, 210 retry
  [Batch 2] 210 windows... WARNING 06-05 05:04:07 [scheduler.py:176

In [ ]:
from google.colab import files
files.download("Temp_Results_microsoft_Phi-3.5-mini-instruct.csv")